# Notebook 01: Ingesta de Datos Sísmicos y Cálculo de Posición Lunar

**Proyecto:** Lunar Tidal Triggering of Earthquakes (Ecuación IvAndreM)  
**Autor:** Iván Andrés Mena Contreras  
**Fecha:** 2026-04-21

## Objetivo
Este notebook hace dos cosas:
1. Descarga terremotos de magnitud >7.0 desde la API pública del USGS
2. Para los primeros 20 eventos, calcula dónde estaba la Luna exactamente en ese momento usando NASA JPL Horizons

Al final guardamos todo en un CSV que servirá de base para el análisis posterior.

In [1]:
# === Imports principales ===
# requests : peticiones HTTP a la API del USGS (descarga del catálogo sísmico)
# pandas   : manejo tabular (DataFrames, CSVs)
# numpy    : operaciones numéricas y manejo de NaN
import requests
import pandas as pd
import numpy as np

# astropy.time.Time: convierte fechas/timestamps a Julian Date,
# que es el formato que JPL Horizons necesita para ubicar a la Luna.
from astropy.time import Time

# astroquery.jplhorizons.Horizons: cliente que consulta el sistema
# JPL Horizons de la NASA para obtener efemérides (posiciones astronómicas).
from astroquery.jplhorizons import Horizons

# datetime: utilidades para fechas legibles
from datetime import datetime

# Silenciamos warnings ruidosos para que la salida sea legible.
# Las excepciones reales seguirán saltando con normalidad.
import warnings
warnings.filterwarnings('ignore')

print("Librerías cargadas correctamente ✓")

Librerías cargadas correctamente ✓


## 📡 Sección 1: Descarga de terremotos desde USGS

Vamos a usar la **API ComCat** del USGS (United States Geological Survey),
un servicio público que permite consultar el catálogo mundial de terremotos.

**Endpoint:** `https://earthquake.usgs.gov/fdsnws/event/1/query`

**Parámetros que usamos:**
- `format=geojson` → respuesta en formato GeoJSON (JSON geoespacial estándar)
- `minmagnitude=7.0` → solo eventos grandes (M ≥ 7.0), los más relevantes
  para buscar una posible influencia lunar
- `maxdepth=70` → solo terremotos someros (≤ 70 km), donde el efecto de
  marea cortical es más plausible
- `starttime` / `endtime` → ventana temporal 1995–2024 (30 años de datos)

In [2]:
# --- Descarga del catálogo de terremotos M>=7.0 desde USGS ---

# URL base del servicio FDSN del USGS
BASE_URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"

# Diccionario de parámetros (requests lo convertirá en query-string).
parametros = {
    'format':       'geojson',     # tipo de respuesta
    'minmagnitude': 7.0,           # magnitud mínima
    'maxdepth':     70,            # profundidad máxima en km (eventos someros)
    'starttime':    '1995-01-01',  # inicio del rango (inclusivo)
    'endtime':      '2024-12-31',  # fin del rango (inclusivo)
}

print("Descargando catálogo de terremotos desde USGS... (puede tardar unos segundos)")

# Petición HTTP GET. timeout=60 evita que se cuelgue indefinidamente.
response = requests.get(BASE_URL, params=parametros, timeout=60)

# Validación: 200 = OK. Si no, abortamos con un mensaje claro.
if response.status_code != 200:
    raise RuntimeError(
        f"Error al consultar USGS. Código HTTP: {response.status_code}\n"
        f"Cuerpo (primeros 300 chars): {response.text[:300]}"
    )

# El cuerpo de la respuesta es JSON; lo convertimos a un dict de Python.
data = response.json()

# data['features'] es la lista de terremotos individuales.
n_eventos = len(data.get('features', []))
print(f"✓ Se encontraron {n_eventos} terremotos M>7.0 entre 1995 y 2024")

Descargando catálogo de terremotos desde USGS... (puede tardar unos segundos)


✓ Se encontraron 335 terremotos M>7.0 entre 1995 y 2024


In [3]:
# --- Parseo del GeoJSON a un DataFrame de pandas ---
# Cada elemento de data['features'] es un terremoto con dos sub-objetos clave:
#   - 'properties' → magnitud, lugar, tiempo, tipo, etc.
#   - 'geometry'   → coordenadas [longitud, latitud, profundidad]

filas = []  # acumulamos un dict por terremoto

for feature in data['features']:
    props = feature['properties']
    geom  = feature['geometry']
    coords = geom['coordinates']  # formato GeoJSON: [lon, lat, depth_km]

    # 'time' viene en milisegundos desde 1970-01-01 UTC (epoch).
    # pandas.to_datetime con unit='ms' y utc=True lo convierte a un
    # datetime con zona horaria UTC.
    tiempo_dt = pd.to_datetime(props['time'], unit='ms', utc=True)

    filas.append({
        'id':        feature['id'],
        'time':      tiempo_dt,
        'latitude':  coords[1],
        'longitude': coords[0],
        'depth':     coords[2],   # profundidad en km
        'magnitude': props['mag'],
        'place':     props['place'],
        'type':      props['type'],
    })

# Construimos el DataFrame y ordenamos cronológicamente (ascendente).
df = pd.DataFrame(filas).sort_values('time').reset_index(drop=True)

print("Primeros 5 terremotos del catálogo:")
print(df.head(5))
print()
print("Información estructural del DataFrame:")
df.info()

Primeros 5 terremotos del catálogo:
           id                             time  latitude  longitude  depth  \
0  usp0006qyz 1995-01-06 22:37:34.320000+00:00    40.246    142.175   26.9   
1  usp0006sef 1995-02-05 22:51:05.140000+00:00   -37.759    178.752   21.1   
2  usp0006vb3 1995-04-07 22:06:56.890000+00:00   -15.199   -173.529   21.2   
3  usp0006vz9 1995-04-21 00:34:46.090000+00:00    12.059    125.580   20.7   
4  usp0006ww5 1995-05-05 03:53:45.050000+00:00    12.626    125.297   16.0   

   magnitude                                place        type  
0        7.0        63 km ESE of Hachinohe, Japan  earthquake  
1        7.1  118 km NNE of Gisborne, New Zealand  earthquake  
2        7.4           88 km NNE of Hihifo, Tonga  earthquake  
3        7.2       10 km E of Dapdap, Philippines  earthquake  
4        7.1    14 km NE of Cabatuan, Philippines  earthquake  

Información estructural del DataFrame:
<class 'pandas.DataFrame'>
RangeIndex: 335 entries, 0 to 334
Data colum

## 🌙 Sección 2: Cálculo de posición lunar con NASA Horizons

**JPL Horizons** es el servicio de efemérides del Jet Propulsion Laboratory (NASA).
Dado un cuerpo (la Luna = id `'301'`) y un instante (en *Julian Date*), nos devuelve
con alta precisión su posición en el cielo y su distancia al observador.

### Conceptos clave
- **Julian Date (JD):** sistema continuo que cuenta días desde el 1 de enero
  del año 4713 a.C. (mediodía UTC). Ej.: `2451545.0` = 1 enero 2000, 12:00 UTC.
  Es el estándar astronómico porque no tiene saltos de calendario ni años bisiestos.
- **RA (Right Ascension / Ascensión Recta):** análogo astronómico de la longitud,
  medida en grados (también suele expresarse en horas).
- **Dec (Declination / Declinación):** análogo de la latitud, en grados.
- **Distancia (delta):** distancia Tierra–Luna en *unidades astronómicas (AU)*.
  La Luna está a ≈ 0.00257 AU (≈ 384 000 km) en promedio.

Como cada llamada a Horizons es una petición de red, en este primer notebook
solo procesamos los **primeros 20 eventos** del catálogo (sondeo rápido).

In [4]:
# --- Cálculo de la posición lunar para los primeros 20 terremotos ---
# Tomamos una copia para no contaminar el DataFrame original.
df_sample = df.head(20).copy()

# Inicializamos las nuevas columnas con NaN. Si una llamada a Horizons
# falla, esa fila queda con NaN y podremos detectarla más tarde.
df_sample['julian_date']      = np.nan
df_sample['moon_ra']          = np.nan   # Ascensión Recta de la Luna (grados)
df_sample['moon_dec']         = np.nan   # Declinación de la Luna (grados)
df_sample['moon_distance_au'] = np.nan   # Distancia Tierra–Luna (AU)

print(f"Consultando JPL Horizons para {len(df_sample)} eventos...\n")

for i, row in df_sample.iterrows():
    try:
        # 1) Fecha del terremoto (datetime UTC) -> Julian Date.
        #    astropy.time.Time acepta un datetime con tz=UTC directamente.
        t  = Time(row['time'].to_pydatetime(), scale='utc')
        jd = float(t.jd)

        # 2) Objeto Horizons:
        #    - id='301'       → la Luna (código NAIF/JPL)
        #    - location='500' → observador en el geocentro de la Tierra
        #    - epochs=jd      → instante exacto requerido
        obj = Horizons(id='301', location='500', epochs=jd)

        # 3) ephemerides() devuelve una tabla astropy con muchas columnas;
        #    para un único epoch nos quedamos con la primera fila.
        eph = obj.ephemerides()
        ra    = float(eph['RA'][0])     # grados
        dec   = float(eph['DEC'][0])    # grados
        delta = float(eph['delta'][0])  # AU (distancia Tierra–Luna)

        # 4) Volcamos los resultados al DataFrame.
        df_sample.at[i, 'julian_date']      = jd
        df_sample.at[i, 'moon_ra']          = ra
        df_sample.at[i, 'moon_dec']         = dec
        df_sample.at[i, 'moon_distance_au'] = delta

        place_short = (row['place'] or '')[:40]
        print(f"✓ Evento {i+1}/{len(df_sample)} procesado: {place_short}")

    except Exception as e:
        # Si Horizons falla (red, rate limit, etc.) seguimos con el resto
        # pero dejamos constancia en la salida para diagnosticar después.
        print(f"✗ Evento {i+1} falló: {type(e).__name__}: {e}")

print("\n--- Cálculo lunar terminado ---")

Consultando JPL Horizons para 20 eventos...

✓ Evento 1/20 procesado: 63 km ESE of Hachinohe, Japan
✓ Evento 2/20 procesado: 118 km NNE of Gisborne, New Zealand
✓ Evento 3/20 procesado: 88 km NNE of Hihifo, Tonga
✓ Evento 4/20 procesado: 10 km E of Dapdap, Philippines


✓ Evento 5/20 procesado: 14 km NE of Cabatuan, Philippines

✓ Evento 6/20 procesado: 249 km E of Vao, New Caledonia
✓ Evento 7/20 procesado: 85 km S of Tungor, Russia
✓ Evento 8/20 procesado: Kermadec Islands, New Zealand
✓ Evento 9/20 procesado: 36 km NNE of Antofagasta, Chile
✓ Evento 10/20 procesado: 155 km WNW of Panguna, Papua New Guinea


✓ Evento 11/20 procesado: 139 km WNW of Panguna, Papua New Guinea


✓ Evento 12/20 procesado: 5 km N of Azoyú, Mexico


✓ Evento 13/20 procesado: 45 km SE of Sucúa, Ecuador
✓ Evento 14/20 procesado: 1995 Colima-Jalisco, Mexico Earthquake
✓ Evento 15/20 procesado: 83 km SE of Naze, Japan
✓ Evento 16/20 procesado: 27 km SSE of Nuwaybi‘a, Egypt
✓ Evento 17/20 procesado: 128 km ESE of Kuril’sk, Russia


✓ Evento 18/20 procesado: 181 km N of Palu, Indonesia
✓ Evento 19/20 procesado: 158 km E of Kuril’sk, Russia


✓ Evento 20/20 procesado: 1996 Biak, Indonesia Earthquake

--- Cálculo lunar terminado ---


In [5]:
# --- Persistencia: guardamos el DataFrame enriquecido en CSV ---
# Lo dejamos en data/raw porque es la primera capa de datos
# (descarga + enriquecimiento bruto, sin transformaciones analíticas).
#
# Nota: cuando este notebook se ejecuta, el directorio de trabajo (CWD)
# es la carpeta 'notebooks/', así que subimos un nivel con '..' para
# llegar a la raíz del proyecto. mkdir(parents=True, exist_ok=True)
# crea la carpeta si no existe (idempotente: no falla si ya existe).
from pathlib import Path

ruta_dir = Path('..') / 'data' / 'raw'
ruta_dir.mkdir(parents=True, exist_ok=True)
ruta_csv = ruta_dir / 'earthquakes_with_moon_sample.csv'

df_sample.to_csv(ruta_csv, index=False)

print(f"✓ CSV guardado con {len(df_sample)} eventos en: {ruta_csv.resolve()}")

✓ CSV guardado con 20 eventos en: C:\Users\IVAN MENA\Documents\lunar-seismic-triggering\data\raw\earthquakes_with_moon_sample.csv


In [6]:
# --- Resumen estadístico final ---
print("=" * 60)
print("ESTADÍSTICAS DE LA MUESTRA PROCESADA")
print("=" * 60)
print(f"Eventos totales en la muestra : {len(df_sample)}")
print(f"Rango de fechas               : {df_sample['time'].min()}  →  {df_sample['time'].max()}")
print(f"Magnitud  (min / max / media) : {df_sample['magnitude'].min():.2f} / {df_sample['magnitude'].max():.2f} / {df_sample['magnitude'].mean():.2f}")
print(f"Profundidad km (min/max/media): {df_sample['depth'].min():.2f} / {df_sample['depth'].max():.2f} / {df_sample['depth'].mean():.2f}")
print()
print("Primeras 5 filas del DataFrame final:")
print(df_sample.head(5))
print()
print("🎉 Primer notebook completado. Primera victoria del proyecto IvAndreM.")

ESTADÍSTICAS DE LA MUESTRA PROCESADA
Eventos totales en la muestra : 20
Rango de fechas               : 1995-01-06 22:37:34.320000+00:00  →  1996-02-17 05:59:30.550000+00:00
Magnitud  (min / max / media) : 7.00 / 8.09 / 7.42
Profundidad km (min/max/media): 10.00 / 45.60 / 26.62

Primeras 5 filas del DataFrame final:
           id                             time  latitude  longitude  depth  \
0  usp0006qyz 1995-01-06 22:37:34.320000+00:00    40.246    142.175   26.9   
1  usp0006sef 1995-02-05 22:51:05.140000+00:00   -37.759    178.752   21.1   
2  usp0006vb3 1995-04-07 22:06:56.890000+00:00   -15.199   -173.529   21.2   
3  usp0006vz9 1995-04-21 00:34:46.090000+00:00    12.059    125.580   20.7   
4  usp0006ww5 1995-05-05 03:53:45.050000+00:00    12.626    125.297   16.0   

   magnitude                                place        type   julian_date  \
0        7.0        63 km ESE of Hachinohe, Japan  earthquake  2.449724e+06   
1        7.1  118 km NNE of Gisborne, New Zealand  eart